## Set up Data

#### Mount drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

#### Unzip the Images

In [ ]:
import zipfile, os

CHALLENGE_DIR = "/content/drive/MyDrive/AML"
ZIP_PATH      = os.path.join(CHALLENGE_DIR, "dataset.zip")
IMAGES_DIR    = "/content/images"   # extracted here (fast local SSD, not Drive)

if not os.path.exists(IMAGES_DIR):
    print("Extracting images...")
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall(IMAGES_DIR)
    print(f"Done — extracted to {IMAGES_DIR}")
else:
    print(f"Already extracted: {IMAGES_DIR}")

# Peek at what's inside
sample_files = os.listdir(os.path.join(IMAGES_DIR,"images"))[:5]
print(f"Total images : {len(os.listdir(os.path.join(IMAGES_DIR,"images")))}")
print(f"Sample names : {sample_files}")
IMAGES_DIR = os.path.join(IMAGES_DIR,"images")

#### Load the Training Data

In [ ]:
import pandas as pd

train_df = pd.read_csv(os.path.join(CHALLENGE_DIR, "train.csv"))
test_df  = pd.read_csv(os.path.join(CHALLENGE_DIR, "test.csv"))

# Map image_id → full path
def img_path(image_id):
    # Try common extensions; adjust if your images have a fixed extension
    for ext in ["", ".jpg", ".jpeg", ".png", ".webp"]:
        p = os.path.join(IMAGES_DIR, f"{image_id}{ext}")
        if os.path.exists(p):
            return p
    return None

train_df["path"] = train_df["image_id"].apply(img_path)
test_df["path"]  = test_df["image_id"].apply(img_path)

print(f"Train rows   : {len(train_df)}")
print(f"Test  rows   : {len(test_df)}")
print("\nLabel distribution:")
print(train_df["ground_truth"].value_counts().rename({0: "authentic", 1: "ai_generated"}))


## Set Up EfficientNetV2

In [ ]:
model_name = "tf_efficientnetv2_s.in21k"
FOLD_TO_RUN = 0

### Extract Features

In [ ]:
!pip install timm

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image

class AIDetectionDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        img_path = self.dataframe.iloc[idx]['path']
        label = self.dataframe.iloc[idx]['ground_truth']
        image = Image.open(img_path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        return image, label

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

#### From pretrained moel

In [ ]:
import timm

model = timm.create_model(model_name, pretrained=True, num_classes=0)
model = model.to(device)
model.eval()

data_config = timm.data.resolve_model_data_config(model)
preprocess = timm.data.create_transform(**data_config, is_training=False)

In [ ]:
import numpy as np
from tqdm.auto import tqdm

def extract_features(df, batch_size=32):
    dataset = AIDetectionDataset(train_df, transform=preprocess)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=2)

    features_list = []
    labels_list = []

    with torch.no_grad():
        for images, labels in tqdm(dataloader):
            images = images.to(device)

            outputs = model(images)

            features_list.append(outputs.cpu().numpy())
            labels_list.append(labels.numpy())

    return np.vstack(features_list), np.concatenate(labels_list)

X, y = extract_features(train_df)

In [ ]:
FEATURES_PATH = os.path.join(CHALLENGE_DIR, f"features/{model_name}_extracted_features.npy")
LABELS_PATH = os.path.join(CHALLENGE_DIR, f"features/{model_name}_extracted_labels.npy")

np.save(FEATURES_PATH, X)
np.save(LABELS_PATH, y)

print(f"Features saved to: {FEATURES_PATH}")
print(f"Labels saved to: {LABELS_PATH}")

#### From fine tuned model

In [ ]:
from sklearn.model_selection import StratifiedKFold, train_test_split

TOTAL_FOLDS = 10

skf = StratifiedKFold(n_splits=TOTAL_FOLDS, shuffle=True, random_state=42)

folds = list(skf.split(train_df, train_df["ground_truth"]))
train_val_idx, test_idx = folds[FOLD_TO_RUN]

test_data = train_df.iloc[test_idx]
pool_data = train_df.iloc[train_val_idx]
train_data, val_data = train_test_split(
    pool_data,
    test_size=0.111,
    stratify=pool_data["ground_truth"],
    random_state=42
)

print(f"EXPERIMENT RUN: Fold {FOLD_TO_RUN}")
print(f"Training samples:   {len(train_data)}")
print(f"Validation samples: {len(val_data)}")
print(f"Test samples:       {len(test_data)}")
# train_val_data, test_data = train_test_split(train_df, test_size=0.1, stratify=train_df["ground_truth"], random_state=42)
# train_data, val_data = train_test_split(train_val_data, test_size=0.111, stratify=train_val_data["ground_truth"], random_state=42)

In [ ]:
import timm
import torchvision.transforms.v2 as T

model = timm.create_model(model_name, pretrained=False, num_classes=2, drop_rate=0.3, drop_path_rate=0.2)

model_path = os.path.join(CHALLENGE_DIR, f"models/{model_name}_try{FOLD_TO_RUN + 1}.pth")
model.load_state_dict(torch.load(model_path, map_location=device))
model.reset_classifier(0)

model = model.to(device)
model.eval()

In [ ]:
data_config = timm.data.resolve_model_data_config(model)
input_size = data_config['input_size'][-2:]
mean = data_config['mean']
std = data_config['std']

# preprocess = timm.data.create_transform(**data_config, is_training=False)
preprocess = T.Compose([
    T.Resize(input_size),
    T.CenterCrop(input_size),
    T.ToImage(),
    T.ToDtype(torch.float32, scale=True),
    T.Normalize(mean=mean, std=std)
])

train_dataset = AIDetectionDataset(train_data, transform=preprocess)
val_dataset = AIDetectionDataset(val_data, transform=preprocess)
test_dataset = AIDetectionDataset(test_data, transform=preprocess)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=False, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)

In [ ]:
from tqdm.auto import tqdm
import numpy as np

def extract_features(dataloader, model):
    all_features = []
    all_labels = []

    with torch.no_grad():
        for images, labels in tqdm(dataloader, desc="Extracting Features"):
            images = images.to(device)

            with torch.amp.autocast("cuda"):
                features = model(images)

            all_features.append(features.cpu().numpy())
            all_labels.append(labels.numpy())

    X_matrix = np.vstack(all_features)
    y_vector = np.concatenate(all_labels)

    return X_matrix, y_vector

print("Extracting Training Features...")
X_train, y_train = extract_features(train_loader, model)

print("Extracting Validation Features...")
X_val, y_val = extract_features(val_loader, model)

print("Extracting Test Features...")
X_test, y_test = extract_features(test_loader, model)

In [ ]:
TRAIN_FEATURES_PATH = os.path.join(CHALLENGE_DIR, f"features/try{FOLD_TO_RUN + 1}_train_features.npy")
TRAIN_LABELS_PATH = os.path.join(CHALLENGE_DIR, f"features/try{FOLD_TO_RUN + 1}_train_labels.npy")
VAL_FEATURES_PATH = os.path.join(CHALLENGE_DIR, f"features/try{FOLD_TO_RUN + 1}_val_features.npy")
VAL_LABELS_PATH = os.path.join(CHALLENGE_DIR, f"features/try{FOLD_TO_RUN + 1}_val_labels.npy")
TEST_FEATURES_PATH = os.path.join(CHALLENGE_DIR, f"features/try{FOLD_TO_RUN + 1}_test_features.npy")
TEST_LABELS_PATH = os.path.join(CHALLENGE_DIR, f"features/try{FOLD_TO_RUN + 1}_test_labels.npy")

np.save(TRAIN_FEATURES_PATH, X_train)
np.save(TRAIN_LABELS_PATH, y_train)
np.save(VAL_FEATURES_PATH, X_val)
np.save(VAL_LABELS_PATH, y_val)
np.save(TEST_FEATURES_PATH, X_test)
np.save(TEST_LABELS_PATH, y_test)

### Load Features

In [ ]:
import numpy as np

# TRAIN_FEATURES_PATH = os.path.join(CHALLENGE_DIR, f"features/{model_name}_train_features.npy")
# TRAIN_LABELS_PATH = os.path.join(CHALLENGE_DIR, f"features/{model_name}_train_labels.npy")
# VAL_FEATURES_PATH = os.path.join(CHALLENGE_DIR, f"features/{model_name}_val_features.npy")
# VAL_LABELS_PATH = os.path.join(CHALLENGE_DIR, f"features/{model_name}_val_labels.npy")
# TEST_FEATURES_PATH = os.path.join(CHALLENGE_DIR, f"features/{model_name}_test_features.npy")
# TEST_LABELS_PATH = os.path.join(CHALLENGE_DIR, f"features/{model_name}_test_labels.npy")

TRAIN_FEATURES_PATH = os.path.join(CHALLENGE_DIR, f"features/try{FOLD_TO_RUN + 1}_train_features.npy")
TRAIN_LABELS_PATH = os.path.join(CHALLENGE_DIR, f"features/try{FOLD_TO_RUN + 1}_train_labels.npy")
VAL_FEATURES_PATH = os.path.join(CHALLENGE_DIR, f"features/try{FOLD_TO_RUN + 1}_val_features.npy")
VAL_LABELS_PATH = os.path.join(CHALLENGE_DIR, f"features/try{FOLD_TO_RUN + 1}_val_labels.npy")
TEST_FEATURES_PATH = os.path.join(CHALLENGE_DIR, f"features/try{FOLD_TO_RUN + 1}_test_features.npy")
TEST_LABELS_PATH = os.path.join(CHALLENGE_DIR, f"features/try{FOLD_TO_RUN + 1}_test_labels.npy")

X_train = np.load(TRAIN_FEATURES_PATH)
X_val = np.load(VAL_FEATURES_PATH)
X_test = np.load(TEST_FEATURES_PATH)
y_train = np.load(TRAIN_LABELS_PATH)
y_val = np.load(VAL_LABELS_PATH)
y_test = np.load(TEST_LABELS_PATH)

print(X_train.shape, X_val.shape, X_test.shape, y_train.shape, y_val.shape, y_test.shape)

## SVM

In [ ]:
from sklearn.model_selection import GridSearchCV, PredefinedSplit
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, f1_score, roc_auc_score, roc_curve
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

X_search_scaled = np.vstack((X_train_scaled, X_val_scaled))
y_search = np.concatenate((y_train, y_val))

split_index = np.concatenate((
    -1 * np.ones(len(X_train)),
     0 * np.ones(len(X_val))
))
pds = PredefinedSplit(test_fold=split_index)

#### Find best param

In [ ]:
classifier = SVC()

In [ ]:
param_grid = {
    'C': [0.1, 1, 10],
}

grid = GridSearchCV(classifier, param_grid, cv=pds, scoring='f1')
grid.fit(X_search_scaled, y_search)

print(grid.best_params_, grid.best_score_)

#### Straight

In [ ]:
classifier = SVC(C=2, probability=True)
classifier.fit(X_train_scaled, y_train)

#### Prediction and report

In [ ]:
y_pred = classifier.predict(X_test_scaled)

print(f"\nSVM Classification Report (try {FOLD_TO_RUN + 1}):")
print(classification_report(y_test, y_pred, target_names=["Authentic", "AI-Generated"]))
print(f"F1: {f1_score(y_test, y_pred):.3f}")

In [ ]:
y_test_probs = classifier.predict_proba(X_test_scaled)[:, 1]
auc_score = roc_auc_score(y_test, y_test_probs)
fpr, tpr, _ = roc_curve(y_test, y_test_probs)

plt.figure(figsize=(3.5, 3))
plt.plot(fpr, tpr, color='darkorange', lw=1.5, label=f'Proposed (AUC={auc_score:.3f})')
plt.plot([0, 1], [0, 1], color='navy', lw=1, linestyle='--', label='Random')

plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate', fontsize=9)
plt.ylabel('True Positive Rate', fontsize=9)
plt.xticks(fontsize=8)
plt.yticks(fontsize=8)
plt.legend(loc="lower right", fontsize=8)
plt.grid(True, alpha=0.2)

plt.savefig('roc_curve_final.pdf', format='pdf', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
plt.rcParams['font.size'] = 8

fig, ax = plt.subplots(figsize=(3.5, 3))

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Authentic', 'AI'])
disp.plot(cmap='Blues', ax=ax, colorbar=True)

ax.set_xlabel('Predicted Label', fontsize=10)
ax.set_ylabel('True Label', fontsize=10)

plt.savefig('svm_confusion_matrix.pdf', format='pdf', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
from PIL import Image

distances = classifier.decision_function(X_test_scaled)
errors_mask = (y_test != y_pred)
error_indices = np.where(errors_mask)[0]
error_confidences = np.abs(distances[errors_mask])

top_errors = error_indices[np.argsort(error_confidences)[::-1]]

print("Top 5 Most Confident Mistakes:\n")

for idx in top_errors[:5]:
    actual = "AI" if y_test[idx] == 1 else "Real"
    pred = "AI" if y_pred[idx] == 1 else "Real"
    conf = abs(distances[idx])

    img_path = train_df.iloc[idx]['path']
    img = Image.open(img_path)

    plt.figure(figsize=(5, 5))
    plt.imshow(img)
    plt.title(f"Actual: {actual} | Predicted: {pred} | Confidence Score: {conf:.2f}")
    plt.axis('off')
    plt.show()

## Inference on Test Set

In [ ]:
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from PIL import Image

class AIInferenceDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        img_path = self.dataframe.iloc[idx]['path']
        image = Image.open(img_path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        return image

submission_dataset = AIInferenceDataset(test_df, transform=preprocess)
submission_dataloader = DataLoader(submission_dataset, batch_size=32, shuffle=False, num_workers=2)

model.eval()
sub_features_list = []

with torch.no_grad():
    for images in tqdm(submission_dataloader):
        images = images.to(device)
        with torch.amp.autocast("cuda"):
            outputs = model(images)
        sub_features_list.append(outputs.cpu().numpy())

X_submission = np.vstack(sub_features_list)

probabilities = classifier.predict_proba(X_submission)[:, 1]

fold_results = pd.DataFrame({
    'image_id': test_df['image_id'],
    'prob': probabilities
})

output_filename = f'fold{FOLD_TO_RUN}_probs.csv'
fold_results.to_csv(output_filename, index=False)

print(f"Saved probabilities to {output_filename}")

In [ ]:
f1 = pd.read_csv('fold0_probs.csv')['prob']
f2 = pd.read_csv('fold1_probs.csv')['prob']
f5 = pd.read_csv('fold2_probs.csv')['prob']
f3 = pd.read_csv('fold3_probs.csv')['prob']
f4 = pd.read_csv('fold4_probs.csv')['prob']

final_avg_prob = (f1 + f2 + f3 + f4 + f5) / 5

final_preds = (final_avg_prob >= 0.5).astype(int)

submission_df = pd.DataFrame({
    'image_id': test_df['image_id'],
    'prediction': final_preds
})
submission_df.to_csv('submission.csv', index=False)